# E037 — GMM Covariance Type Ablation

**Motivation:** All experiments used diagonal covariance GMMs (`covariance_type='diag'`). Full covariance (`'full'`) or tied covariance (`'tied'`) may capture feature correlations better, potentially improving discrimination.

**Trade-off:** Full covariance has O(d²) parameters vs O(d) for diagonal. With d=39 and ~5400 training frames, full covariance (1521 params/component) may overfit vs diagonal (39 params/component).

**Hypothesis:** Tied covariance (shared full covariance across components) will improve over diagonal by capturing correlations while controlling parameter count.

**Configs:**
- `diag`: E025 baseline (diagonal, 39 params/comp)
- `full`: Full covariance per component (1521 params/comp)
- `tied`: Shared full covariance (1521 params total)
- `spherical`: Simplest (1 param/comp)

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
import librosa
from scipy.special import logsumexp
from sklearn.mixture import GaussianMixture
import copy

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

SEED = 67
UBM_COMPONENTS = 32
MAP_R = 16.0

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
y_all = manifest['label'].to_numpy()
print(f'{len(manifest)} samples')

E025_REF = {'mean_eer': 1.94, 'std_eer': 1.57}

In [ ]:
def _find_wav(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".wav")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _extract_lpcc(y, sr, order=12, n_cep=13, hop_length=160, win_length=400):
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    cep_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            A_freq = np.fft.rfft(a, n=512)
            log_H = -np.log(np.abs(A_freq) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except Exception:
            cep = np.zeros(n_cep)
        cep_frames.append(cep)
    feat = np.array(cep_frames, dtype=np.float32)
    delta = librosa.feature.delta(feat.T).T
    delta2 = librosa.feature.delta(feat.T, order=2).T
    feat = np.hstack([feat, delta, delta2])
    feat -= feat.mean(axis=0)
    return feat

def _aug_pitch(y, sr, rng):
    n_steps = float(rng.choice([-2, -1, 1, 2]))
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

def train_ubm_map(X_target, X_nontarget, cov_type, seed):
    """Train UBM with specified covariance type and MAP adapt."""
    # Train UBM
    ubm = GaussianMixture(
        n_components=UBM_COMPONENTS,
        covariance_type=cov_type,
        max_iter=200,
        random_state=seed
    ).fit(X_nontarget)
    
    # MAP adaptation (only works properly for diag/full, simplified for others)
    log_resp = ubm._estimate_log_prob(X_target) + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp = np.exp(log_resp)
    n_k = resp.sum(axis=0)
    mu_hat = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha = n_k / (n_k + MAP_R)
    
    adapted = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    
    return ubm, adapted

def train_and_evaluate(train_df, val_df, data_dir, cov_type, seed):
    """Train UBM-MAP and evaluate on validation."""
    rng = np.random.default_rng(seed)
    
    # Extract features
    X_target, X_nontarget = [], []
    for _, row in train_df.iterrows():
        y_wav, sr = librosa.load(_find_wav(row["stem"], data_dir), sr=None, mono=True)
        wavs = [y_wav]
        if row["label"] == 1:  # Target: add pitch aug
            wavs.append(_aug_pitch(y_wav, sr, rng))
        
        for y_aug in wavs:
            feat = _extract_lpcc(y_aug, sr)
            if row["label"] == 1:
                X_target.append(feat)
            else:
                X_nontarget.append(feat)
    
    X_target = np.vstack(X_target)
    X_nontarget = np.vstack(X_nontarget)
    
    # Train
    try:
        ubm, adapted = train_ubm_map(X_target, X_nontarget, cov_type, seed)
    except Exception as e:
        print(f"  Warning: {cov_type} failed: {e}")
        return None, None
    
    # Score validation
    scores = []
    for _, row in val_df.iterrows():
        y_wav, sr = librosa.load(_find_wav(row["stem"], data_dir), sr=None, mono=True)
        feat = _extract_lpcc(y_wav, sr)
        score = adapted.score(feat) - ubm.score(feat)
        scores.append(score)
    
    return np.array(scores), ubm

print('Model functions defined')

## 2. Cross-validation

In [ ]:
cov_types = ['diag', 'full', 'tied', 'spherical']
results = {}

for cov_type in cov_types:
    print(f"\n=== {cov_type} ===")
    fold_eers = []
    
    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
        seed_f = SEED + fold_id
        train_df = manifest.loc[train_idx]
        val_df = manifest.loc[val_idx]
        
        scores, _ = train_and_evaluate(train_df, val_df, DATA, cov_type, seed_f)
        
        if scores is None:
            print(f"  Fold {fold_id}: FAILED")
            continue
        
        eer, _ = compute_eer(scores[y_all[val_idx] == 1], scores[y_all[val_idx] == 0])
        fold_eers.append(eer * 100)
        print(f"  Fold {fold_id}: EER={eer*100:.2f}%")
    
    if len(fold_eers) > 0:
        results[cov_type] = {
            'fold_eers': fold_eers,
            'mean': np.mean(fold_eers),
            'std': np.std(fold_eers),
        }
        print(f"  Mean ± Std: {np.mean(fold_eers):.2f} ± {np.std(fold_eers):.2f}%")
    else:
        print(f"  {cov_type}: All folds failed")

print("\n=== Summary ===")
for cov_type, res in results.items():
    print(f"{cov_type:12s}: {res['mean']:5.2f} ± {res['std']:5.2f}%")

## 3. Conclusion

Covariance type effect: [TBD]
Decision: [Adopt/Reject]